# Herramienta de Extracción de Frases (N-gramas) de Youtube
Este notebook contiene la implementación para la extracción de transcripciones y análisis de frases frecuentes en videos de Youtube.

In [ ]:
!pip install youtube-transcript-api nltk

In [ ]:
import json
import queue
import re
import concurrent.futures
from youtube_transcript_api import YouTubeTranscriptApi
from collections import defaultdict, Counter
import nltk
from nltk.corpus import stopwords
import string

# Descargar las stopwords si no están ya disponibles
nltk.download('stopwords')
stop_words = set(stopwords.words('spanish')) # Se asume español, pero se puede modificar


## Etapa 1: Descarga de Transcripciones (En Paralelo)

En esta etapa se obtienen las transcripciones de las URLs proporcionadas usando `youtube-transcript-api` y se guardan en archivos JSON temporales. Los identificadores de estos archivos se encolan para su posterior procesamiento.

In [ ]:
# Cola nativa de Python para pasar información a la etapa 2
cola_procesamiento = queue.Queue()

def extraer_video_id(url):
    """Extrae el ID del video de YouTube desde la URL."""
    match = re.search(r'(?:v=|\/)([0-9A-Za-z_-]{11}).*', url)
    return match.group(1) if match else None

def procesar_video(url, creador):
    """Descarga la transcripción del video y la guarda en un JSON temporal."""
    video_id = extraer_video_id(url)
    if not video_id:
        print(f"Error: No se pudo extraer el ID de la URL: {url}")
        return

    try:
        # Extraer transcripción nativa (lista de diccionarios con text, start, duration)
        transcripcion = YouTubeTranscriptApi.get_transcript(video_id, languages=['es', 'en'])
        
        # Identificador único para el archivo
        identificador = f"{creador.replace(' ', '_')}_{video_id}"
        archivo_salida = f"{identificador}.json"
        
        # Guardar en JSON temporal
        data = {
            "creador": creador,
            "url": url,
            "video_id": video_id,
            "transcripcion": transcripcion
        }
        
        with open(archivo_salida, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
            
        print(f"Éxito: Transcripción de '{creador}' guardada en {archivo_salida}")
        
        # Enviar identificador a la cola
        cola_procesamiento.put(identificador)
        
    except Exception as e:
        print(f"Error procesando {url} (Creador: {creador}): {str(e)}")

def ejecutar_etapa_1(diccionario_urls, num_hilos=4):
    """Ejecuta la descarga de transcripciones en paralelo."""
    print(f"Iniciando Etapa 1 con {num_hilos} hilos...")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_hilos) as executor:
        # Enviar tareas al pool de hilos
        futuros = [executor.submit(procesar_video, url, creador) for url, creador in diccionario_urls.items()]
        
        # Esperar a que terminen
        concurrent.futures.wait(futuros)
        
    print(f"Etapa 1 finalizada. Archivos en cola: {cola_procesamiento.qsize()}")

# Ejemplo de uso (Puedes modificar este diccionario con tus propios videos)
diccionario_prueba = {
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ": "Rick Astley",
    "https://www.youtube.com/watch?v=jNQXAC9IVRw": "Jawed"
}

# Ejecutar Etapa 1
# ejecutar_etapa_1(diccionario_prueba, num_hilos=2)


## Etapa 2: Procesamiento de N-gramas (En Paralelo)

En esta etapa se procesan los archivos encolados por la Etapa 1. Se extraen los n-gramas más utilizados por cada creador (de 2 a 6 palabras), descartando opcionalmente aquellos que estén compuestos 100% por palabras vacías (stopwords). Se captura la marca de tiempo de la primera aparición del n-grama.

In [ ]:
def limpiar_palabra(palabra):
    """Convierte a minúsculas y elimina signos de puntuación."""
    palabra = palabra.lower()
    return palabra.translate(str.maketrans('', '', string.punctuation))

def es_ngram_valido(ngram_words, usar_filtro_stopwords=True):
    """
    Regla de negocio: Si el filtro está activo, se descarta el n-grama 
    SOLO si TODAS las palabras que lo componen son stopwords.
    """
    if not usar_filtro_stopwords:
        return True
        
    # Validar si todas las palabras están en la lista de stopwords
    son_todas_stopwords = all(palabra in stop_words for palabra in ngram_words)
    
    # Es válido si NO todas son stopwords (es decir, tiene al menos una palabra de valor)
    return not son_todas_stopwords

def tokenizar_transcripcion(transcripcion):
    """
    Descompone la transcripción en una lista de palabras donde cada palabra 
    retiene el inicio (start) y el fin (start + duration) de su bloque original.
    """
    tokens = []
    for bloque in transcripcion:
        texto = bloque.get('text', '')
        start = bloque.get('start', 0)
        end = start + bloque.get('duration', 0)
        
        palabras = texto.split()
        for p in palabras:
            p_limpia = limpiar_palabra(p)
            if p_limpia: # Evitar strings vacíos
                tokens.append({
                    'palabra': p_limpia,
                    'start': start,
                    'end': end
                })
    return tokens

def extraer_ngrams_video(identificador, n_frases, usar_filtro_stopwords=True):
    """Procesa un archivo JSON, genera n-gramas de 2 a 6 palabras y devuelve los resultados."""
    archivo = f"{identificador}.json"
    
    try:
        with open(archivo, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        transcripcion = data['transcripcion']
        creador = data['creador']
        url = data['url']
        
        tokens = tokenizar_transcripcion(transcripcion)
        
        # Diccionario para almacenar conteos e información de primera aparición
        # Estructura: { n_size: { "ngrama_str": {"count": int, "start": float, "end": float, "url": str} } }
        resultados_video = {i: {} for i in range(2, 7)}
        
        total_tokens = len(tokens)
        
        for n in range(2, 7): # N-gramas de tamaño 2, 3, 4, 5, 6
            for i in range(total_tokens - n + 1):
                ngram_tokens = tokens[i:i+n]
                ngram_words = tuple(t['palabra'] for t in ngram_tokens)
                
                # Aplicar regla de negocio (descartar si es 100% stopwords)
                if not es_ngram_valido(ngram_words, usar_filtro_stopwords):
                    continue
                    
                ngram_str = " ".join(ngram_words)
                
                # El start es el del primer token, el end es el del último token
                start_time = ngram_tokens[0]['start']
                end_time = ngram_tokens[-1]['end']
                
                if ngram_str in resultados_video[n]:
                    resultados_video[n][ngram_str]['count'] += 1
                else:
                    resultados_video[n][ngram_str] = {
                        'count': 1,
                        'start': start_time,
                        'end': end_time,
                        'url': url
                    }
                    
        return creador, resultados_video
        
    except Exception as e:
        print(f"Error procesando el archivo {archivo}: {str(e)}")
        return None, None

def fusionar_resultados(resultados_globales, creador, resultados_video):
    """Suma los conteos de los n-gramas de un video a los totales del creador."""
    if creador not in resultados_globales:
        resultados_globales[creador] = {i: {} for i in range(2, 7)}
        
    for n in range(2, 7):
        for ngram_str, info in resultados_video[n].items():
            if ngram_str in resultados_globales[creador][n]:
                resultados_globales[creador][n][ngram_str]['count'] += info['count']
                # Mantenemos el start/end/url de la primera aparición (no lo sobrescribimos)
            else:
                resultados_globales[creador][n][ngram_str] = info

def ejecutar_etapa_2(n_frases=10, num_hilos=4, usar_filtro_stopwords=True):
    """Vacia la cola, procesa en paralelo y extrae el Top N de frases."""
    elementos_procesar = []
    
    # Vaciar la cola
    while not cola_procesamiento.empty():
        elementos_procesar.append(cola_procesamiento.get())
        
    if not elementos_procesar:
        print("La cola está vacía. Ejecuta la Etapa 1 primero.")
        return
        
    print(f"Iniciando Etapa 2: Procesando {len(elementos_procesar)} videos con {num_hilos} hilos...")
    
    resultados_globales = {} # { creador: { n: { ngrama: info } } }
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_hilos) as executor:
        # Enviar tareas
        futuros = [executor.submit(extraer_ngrams_video, ident, n_frases, usar_filtro_stopwords) for ident in elementos_procesar]
        
        # Recopilar resultados conforme terminen
        for futuro in concurrent.futures.as_completed(futuros):
            creador, resultados_video = futuro.result()
            if creador and resultados_video:
                fusionar_resultados(resultados_globales, creador, resultados_video)
                
    # Extraer el Top N por creador
    top_resultados = {}
    for creador, ngrams_por_tamanio in resultados_globales.items():
        top_resultados[creador] = {}
        for n in range(2, 7):
            # Ordenar por conteo (de mayor a menor)
            ordenados = sorted(ngrams_por_tamanio[n].items(), key=lambda x: x[1]['count'], reverse=True)
            # Tomar el top N
            top_resultados[creador][n] = ordenados[:n_frases]
            
    print("Etapa 2 finalizada.")
    return top_resultados

def mostrar_resultados(top_resultados):
    """Formatea y muestra los resultados obtenidos."""
    if not top_resultados:
        return
        
    for creador, resultados in top_resultados.items():
        print(f"\n{'='*50}")
        print(f"CREADOR: {creador}")
        print(f"{'='*50}")
        
        for n in range(2, 7):
            print(f"\n--- Top N-gramas de {n} palabras ---")
            ngrams = resultados.get(n, [])
            if not ngrams:
                print("No se encontraron resultados.")
                continue
                
            for i, (frase, info) in enumerate(ngrams, 1):
                count = info['count']
                start = info['start']
                end = info['end']
                url = info['url']
                print(f"{i}. '{frase}' (Repeticiones: {count})")
                print(f"   Primera aparición: {start:.2f}s - {end:.2f}s | Origen: {url}")

# Ejecutar Etapa 2 y mostrar resultados
# resultados = ejecutar_etapa_2(n_frases=10, num_hilos=2, usar_filtro_stopwords=True)
# mostrar_resultados(resultados)
